# Folder: **H2**

In [ ]:
######################################## Parameters

### Spatial domain 'ES' or 'EU' (for maps domain)
spatial_domain = 'ES'

### Circle and label style
marker_scale = 7500

### Font sizes
title_fontsize = 22
legend_fontsize = 14
circle_id_fontsize = 12
reference_fontsize = 14

In [ ]:
##### Import packages
import os
import sys
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
warnings.filterwarnings('ignore', category=UserWarning)

## `H2_valley_demands.yaml`

Load H2 valley demands and build a table with valley name, coordinates, and demand.

In [ ]:
h2_valley_demands = xp.load_file_yaml(
    params,
    filename='H2_valley_demands.yaml',
    location='data_ES',
    folder='H2',
)

valley_records = []
for valley_cfg in h2_valley_demands.values():
    valley_name = valley_cfg['valley_name']
    valley_params = valley_cfg['valley_params']

    valley_records.append({
        'valley_name': valley_name,
        'x': valley_params['x'],
        'y': valley_params['y'],
        'demand': valley_params['demand'],
    })

h2_valleys = pd.DataFrame(valley_records)
h2_valleys

Plot H2 valleys on the map with circle size proportional to demand.

In [ ]:
#################### Figure
fig_size = [12, 12]
crs = ccrs.PlateCarree()

fig, ax = plt.subplots(figsize=fig_size, subplot_kw={'projection': crs})
ax.set_extent(params[f'boundaries_offshore_{spatial_domain}'], crs=ccrs.PlateCarree())


### Add map features (colors from params.yaml -> map_add_features)
xp.map_add_features(ax, params['map_add_features'])


### Add H2 valley demands
h2_valleys_plot = h2_valleys.copy().reset_index(drop=True)
h2_valleys_plot['valley_id'] = h2_valleys_plot.index + 1

total_demand = h2_valleys_plot['demand'].sum()

circle_sizes = h2_valleys_plot['demand'] * marker_scale
ax.scatter(
    h2_valleys_plot['x'],
    h2_valleys_plot['y'],
    s=circle_sizes,
    color='#1f78b4',
    alpha=0.55,
    edgecolors='black',
    linewidths=1.0,
    transform=ccrs.PlateCarree(),
    zorder=6,
    label='H2 valley demand',
)


### Add valley id inside each circle
for _, row in h2_valleys_plot.iterrows():
    ax.text(
        row['x'],
        row['y'],
        f"{int(row['valley_id'])}",
        transform=ccrs.PlateCarree(),
        fontsize=circle_id_fontsize,
        fontweight='bold',
        ha='center',
        va='center',
        color='white',
        zorder=7,
    )


## Add legend text with number -> valley name
legend_lines = [f"{int(row['valley_id'])}: {row['valley_name']}" for _, row in h2_valleys_plot.iterrows()]
legend_text = "Valley index\n" + "\n".join(legend_lines)
ax.text(
    0.02,
    0.98,
    legend_text,
    transform=ax.transAxes,
    fontsize=legend_fontsize,
    ha='left',
    va='top',
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='black'),
    zorder=8,
)


## Add reference circle for 100 ktH2 (0.1 in demand units)
ax.scatter(
    [0.06],
    [0.82],
    s=[0.1 * marker_scale],
    color='#1f78b4',
    alpha=0.55,
    edgecolors='black',
    linewidths=1.0,
    transform=ax.transAxes,
    zorder=8,
)
ax.text(
    0.09,
    0.82,
    '100 ktH2/year',
    transform=ax.transAxes,
    fontsize=reference_fontsize,
    ha='left',
    va='center',
    zorder=8,
)

ax.set_title(f"H2 valley demands ({total_demand:g} MtH2/year)", fontsize=title_fontsize)
plt.tight_layout()